# Import Library

In [12]:
import pandas as pd
import sklearn
import re
import string
import json
from textblob import TextBlob

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Data

In [13]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
train[train['sumber'] == 'Antara'].head()

,id,sumber,kodekat,kodesubkat,kategori,subkategori,katakunci,tanggal,judul_berita,konten_berita,nama_tokoh,jabatan,organisasi,lokasi,alias,kutipan,sentimen_kutipan,sentimen_berita
0,00e3395ef29,Antara,J,J,Informasi dan Komunikasi,Informasi dan Komunikasi,Laporan keuangan,2021-02-05,ASN penyeleweng dana infak Masjid Raya divonis...,Padang (ANTARA) - Oknum Aparatur Sipil Negara ...,['Rinto'],['Hakim Ketua Yose'],[],"['Padang', 'Sumbar', 'Provinsi Sumbar', 'Inspe...","['dibayar paling lama', '¬©']","['""Menyatakan terdakwa terbukti secara sah dan...",Negatif,Negatif
2,01eb3258ed4,Antara,D,D1,Pengadaan Listrik dan Gas,Ketenagalistrikan,Listrik PLN,2021-03-31,Angkasa Pura minta maaf atas mati listrik di ...,Padang (ANTARA) - PT Angkasa Pura II selaku pe...,"['Ikhwan Wahyudi', 'Adi Lazuardi']",[],"['PT Angkasa Pura II', 'PLN', 'BIM', '***1***']","['Padang', 'Sumbar', 'Kota Padang', 'Kabupaten...",['¬©'],"['""Kami menyampaikan permohonan maaf yang sebe...",Negatif,Negatif
9,04adbfa2838,Antara,J,J,Informasi dan Komunikasi,Informasi dan Komunikasi,teknologi,2021-01-06,Digitalisasi aksara daerah perlu dukungan peme...,Jakarta (ANTARA) - Digitalisasi aksara daerah ...,"['Muhammad Yusuf', 'Yusuf', 'Suryanto', 'Ida N...",['Direktur Eksekutif Culuture and Folks for In...,[],"['Jakarta', 'Jawa', 'Indonesia', 'Jawa di', 'Y...",['¬©'],"['""Pasalnya, ketika digitalisasi aksara Jawa b...",Positif,Negatif
10,04df357122e,Antara,J,J,Informasi dan Komunikasi,Informasi dan Komunikasi,Penerbitan,2021-01-31,Dukcapil ganti dokumen kependudukan 3.549 KK k...,Makassar (ANTARA) - Direktorat Jenderal (Ditje...,"['Prof Dr Zudan Arif Fakrulloh SH MH', 'Asep',...","['Dirjen Dukcapil Kemendagri Prof', 'Ketua Tim...",['Direktorat Jenderal (Ditjen) Kependudukan da...,"['Makassar', 'Sulawesi Barat (Sulbar)', 'Provi...",['¬©'],"['""Ini rutin dilakukan Dukcapil, setiap ada be...",Netral,Negatif
14,05fa9e44723,Antara,F,F,Konstruksi,Konstruksi,Proyek pembangunan,2021-03-15,"Dampak Tol Semanan-Sunter, dua SD di Jakpus ak...",untuk lokasi baru untuk kedua SDN tersebut har...,"['Hartono', 'Dhany Sukma', 'Dhany', 'Dwi Gayati']",['Pejabat Pembuat Komitmen Pengadaan Tanah Jal...,"['Ditjen Bina', 'Kementerian PUPR', 'Pemerinta...","['Jakarta', 'Jakarta Pusat', 'DKI Jakarta']","['SDN', '¬©']",['mengatakan SDN 07 Kebon Kosong dan SDN 02 Ke...,Netral,Negatif


In [6]:
test.head(5)

,id,sumber,kodekat,kodesubkat,kategori,subkategori,katakunci,tanggal,judul_berita,konten_berita,nama_tokoh,jabatan,organisasi,lokasi,alias,kutipan,sentimen_kutipan
0,b67b9e3d9b8,Detik,A,A2,"Pertanian, Kehutanan, dan Perikanan",Kehutanan dan Penebangan Kayu,bambu,2021-02-16,"Viral Pemasungan ODGJ di Bandung Barat, Begini...",Bandung Barat - Jagat dunia maya dihebohkan de...,"['Nanang', 'Bunyamin', 'Eli', 'Lihat']",['Kepala Desa Sindangkerta'],[],"['Bandung Barat', 'Desa Sindangkerta', 'Kecama...",['rumah sakit'],"['""Dari usia 19 tahun jadi sepertu ini (ganggu...",Negatif
1,b687d3fba37,Detik,A,A1.a,"Pertanian, Kehutanan, dan Perikanan","Pertanian, Peternakan, Perburuan dan Jasa Pert...",Produksi Kedelai,2021-05-21,"7 Makanan untuk Penderita Anemia, yang Harus D...",Jakarta -\n\nPenderita anemia perlu mengasup m...,[],[],[],['Jakarta'],[],[],Netral
2,b68e2851dd8,Antara,A,A3,"Pertanian, Kehutanan, dan Perikanan",Perikanan,hasil tangkap nelayan,2021-05-20,Kapal Patroli Polri gagalkan pengeboman ikan d...,Keempat pelaku mengaku akan melakukan pengebom...,[],"['anggota KP', 'Kepala Sub Bidang Penegakan Hu...","['Mabes Polri', 'Direktorat Polair Polda Papua...","['Provinsi Papua Barat', 'Papua Barat', 'Kabup...",[],['mengatakan bahwa saat melakukan patroli di p...,Negatif
3,b6914e3beeb,Detik,A,A1.c,"Pertanian, Kehutanan, dan Perikanan","Pertanian, Peternakan, Perburuan dan Jasa Pert...",Produksi Tebu,2021-01-25,"Rapat Bareng DPR, Mentan Dicecar soal Kedelai ...",Jakarta -\n\nKomisi IV DPR RI hari ini menggel...,['Sudin'],"['Menteri Pertanian (Mentan)', 'Ketua Komisi I...","['Kementerian Pertanian (Kementan)', 'Kementer...","['Jakarta', 'Jakarta Pusat', 'Australia']","['Impor', 'impor']","['""Rapat ini kami buka dan dinyatakan terbuka ...",Negatif
4,b6916aab106,Antara,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,wabah penyakit,2021-01-21,Berkas kasus RS UMMI diserahkan tahap I ke Kej...,Berkas perkara RS Ummi sudah tahap satu kemari...,"['Agung', 'Andi Rian Djajadi', 'Rizieq Shihab'...",['Direktur Tindak Pidana Umum Bareskrim Polri ...,"['Direktorat Tindak Pidana Umum', 'Bareskrim P...","['Berkas', 'Jakarta', 'Bogor', 'Jawa Barat']",[],"['""Berkas perkara RS Ummi sudah tahap satu kem...",Negatif


# EDA Awal

In [27]:
from ydata_profiling import ProfileReport
# Buat laporan profiling
profile = ProfileReport(train, title="Profiling Report", explorative=True)

# Tampilkan di notebook (jika pakai Jupyter/Colab)
profile.to_notebook_iframe()

# Simpan ke file HTML
profile.to_file("profiling_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:27<00:00,  1.51s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Drop Column

In [14]:
train.columns

Index(['id', 'sumber', 'kodekat', 'kodesubkat', 'kategori', 'subkategori',
       'katakunci', 'tanggal', 'judul_berita', 'konten_berita', 'nama_tokoh',
       'jabatan', 'organisasi', 'lokasi', 'alias', 'kutipan',
       'sentimen_kutipan', 'sentimen_berita'],
      dtype='object')

In [15]:
cols = ['kodesubkat', 'kategori', 'subkategori', 'katakunci', 'tanggal', 'nama_tokoh', 'jabatan', 'organisasi', 'lokasi', 'alias']

train.drop(cols, axis=1, inplace=True)
test.drop(cols, axis=1, inplace=True)

In [19]:
train[train['sumber'] == 'Detik'].head(2)

id sumber kodekat  \
1  019a47ed0bc  Detik       A   
4  026dd5917f6  Detik       J   

                                                                    judul_berita  \
1  10 Alasan MK Kategorikan Ganja Hidroponik sebagai Pohon dan Kena UU Narkotika   
4                           Bill Gates Kuasai Tanah Pertanian, Netizen Ketakutan   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

# Text Cleaning

In [25]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 1. Judul Berita

In [16]:
def clean_judul(text):
    text = re.sub(r' [^A-Za-z0-9\']', '', text) #hapus karakter selain huruf, angka sm tanda
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

train['judul_berita'] = train['judul_berita'].apply(clean_judul)
test['judul_berita'] = test['judul_berita'].apply(clean_judul)

## 2. Konten Berita

In [7]:
train['sumber'].value_counts()

sumber
Detik      27097
Antara     18224
Okezone     7568
Name: count, dtype: int64

In [33]:
def clean_detik(text):
    # ubah ke huruf kecil
    text = text.lower()

    # hapus pola lokasi di awal kalimat, misalnya "jakarta - ", "jayapura - ", "australia - "
    text = re.sub(r'^[a-zéèáàäâõóòöôç\s]+-\s+', '', text, flags=re.IGNORECASE)

    # hapus bagian "baca juga:" atau "lihat juga:" beserta teks di antara \n\n sampai \n\n
    text = re.sub(r'(baca juga:|lihat juga:).*?\n\n.*?\n\n', ' ', text, flags=re.DOTALL | re.IGNORECASE)

    # hapus pola tanggal seperti (12/3/22), (1/12/2021)
    text = re.sub(r'\(\d{1,2}/\d{1,2}/\d{2,4}\)', '', text)

    # hapus string \n\n dan variasinya
    text = re.sub(r'\\n+|\n+', ' ', text)

    # hapus pola promosi halaman dan video khas detik.com
    text = re.sub(r'selengkapnya ada di halaman berikutnya>>\s*selanjutnya', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'baca juga:.*?simak video.*?\[gambas:video.*?\]', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'\(fykfyk\)', '', text)

    # hapus kata dalam kurung beserta kurungnya
    text = re.sub(r'\([^)]*\)', ' ', text)

    # hapus link/url (http, https, www)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # hapus baris atau kalimat yang diawali dengan pola wartawan/editor seperti:
    # "penulis:", "editor:", "reporter:", "oleh:", "pewarta:"
    text = re.sub(r'\b(penulis|editor|reporter|oleh|pewarta)\s*[:\-]?\s*[a-z\s\.]+', ' ', text, flags=re.IGNORECASE)

    # tambahan: hapus jika muncul di baris terpisah seperti "\nEditor: Amelia Syahla\n"
    text = re.sub(r'(^|\n)\s*(penulis|editor|reporter|oleh|pewarta)\s*[:\-]\s*[^\n]+', ' ', text, flags=re.IGNORECASE)
    
    # hapus nama hari (ID & EN)
    hari = r'\b(senin|selasa|rabu|kamis|jumat|jum\'at|sabtu|minggu|ahad|monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b'
    text = re.sub(hari, ' ', text, flags=re.IGNORECASE)

    # hapus nama bulan (ID & EN)
    bulan = r'\b(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\b'
    text = re.sub(bulan, ' ', text, flags=re.IGNORECASE)

    # hapus simbol hak cipta seperti "¬© antara", "©", "¬©", dsb
    text = re.sub(r'¬?©\s*\w*', ' ', text)
    text = re.sub(r'\bantara\b', ' ', text)  # hapus "antara" jika muncul sebagai watermark di akhir teks

    # hapus angka dan simbol tidak penting
    text = re.sub(r'\d+', ' ', text)  # hapus angka
    text = re.sub(r'[“”‘’•–…•—•−·–—_=:;<>|~`^]', ' ', text)  # hapus simbol umum
    text = text.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca standar

    # rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

train['konten_berita'] = train['konten_berita'].apply(clean_detik)

In [ ]:
train['konten_berita'].(1)

52888    kami meyakini keamanan keseluruhan vaksin sinovac karena terbukti aman sebagaimana hasil observasi vaksinasi massal beijing seorang warga hong kong berusia tahun meninggal dunia setelah menerima vaksin covid buatan perusahaan farmasi asal china sinovac biotech namun media china pagi melaporkan bahwa kematian pria lansia tersebut tidak terkait langsung dengan vaksin sinovac dengan mengutip hasil autopsi dan program vaksinasi massal di hong kong terus berlanjut sesuai rencana kematian tidak terkait langsung dengan suntikan vaksin karena hasil autopsi menyimpulkan pria tersebut menderita penyakit kronis obesitas tekanan darah tinggi dan dicurigai menderita penyakit jantung koroner demikian hung fan ngai ivan selaku penanggung jawab klinis vaksinasi massal di wilayah korban dalam konferensi pers di hong kong malam menurut hung kemungkinan besar korban meninggal karena penyakit jantung paruparu dan kegagalan pada sistem pernapasan
Name: konten_berita, dtype: object